In [1]:
import pandas as pd
from churn.features import build_feature_set, encode_categorical_features

customers = pd.read_csv("../data/generated/customers_data.csv")
sales = pd.read_csv("../data/generated/sales_data.csv", parse_dates=["Date"])

reference_date = pd.Timestamp("2025-12-31")

features = build_feature_set(customers, sales, reference_date)
features_encoded = encode_categorical_features(features)

print(features_encoded.shape)

(1000, 18)


In [2]:
from sklearn.model_selection import train_test_split

X = features_encoded.drop(columns=["Customer_ID", "Churn"])
y = features_encoded["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Proportion churn train:", y_train.mean().round(3))
print("Proportion churn test:", y_test.mean().round(3))

Train: (800, 16) Test: (200, 16)
Proportion churn train: 0.408
Proportion churn test: 0.41


### Interprétation — chargement et split

- **(1000, 18)** : rechargement via `build_feature_set` + `encode_categorical_features`
  identique au résultat obtenu dans `eda_churn.ipynb` — confirme que le pipeline
  extrait dans `features.py` est bien reproductible, sans avoir eu à réexpliquer
  ou recoder la logique de leakage/encodage.
- **Train (800, 16) / Test (200, 16)** : split 80/20 comme prévu. 16 colonnes
  = 18 - Customer_ID (identifiant, exclu) - Churn (la cible, séparée dans y).
- **Proportion churn : 0.408 (train) vs 0.41 (test)** — quasi identiques,
  écart de 0.002. Confirme que `stratify=y` a bien fonctionné : le split
  respecte la proportion réelle du dataset (40.8% de churn), pas de biais
  introduit par le split lui-même.

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # apprend ET applique sur train
X_test_scaled = scaler.transform(X_test)         # applique seulement, n'apprend pas

model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

### Entraînement — Logistic Regression (baseline)

Modèle entraîné sur X_train_scaled/y_train (StandardScaler appris sur train
uniquement, appliqué sur test). Pas d'erreur, prêt pour évaluation.

In [5]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

print(classification_report(y_test, y_pred))

y_proba = model.predict_proba(X_test_scaled)[:, 1]
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       118
           1       1.00      0.98      0.99        82

    accuracy                           0.99       200
   macro avg       0.99      0.99      0.99       200
weighted avg       0.99      0.99      0.99       200

ROC-AUC: 0.999896651508888
[[118   0]
 [  2  80]]


### Interprétation — Logistic Regression (baseline)

- **Recall (classe 1, churn) = 0.98** : sur 82 vrais churners dans le test,
  80 détectés, 2 ratés (faux négatifs). Excellent pour notre priorité métier.
- **Precision (classe 1) = 1.00** : sur tous les clients signalés comme
  churners, 100% le sont vraiment — zéro fausse alerte.
- **F1 = 0.99** : cohérent, Precision et Recall sont tous les deux très hauts.
- **ROC-AUC = 0.9999** : quasi séparation parfaite entre les deux classes.

### ⚠️ Limite — performance quasi parfaite due au dataset synthétique

ROC-AUC de 0.9999 n'est pas un signal de "bon modèle" au sens production —
c'est le résultat attendu d'un dataset généré sans bruit réaliste entre
Recency/Frequency et Churn (séparation quasi déterministe dès l'EDA).
Sur des données réelles, un tel score serait suspect et justifierait une
enquête de leakage. À mentionner explicitement dans le rapport (M9) comme
limite du dataset, pas comme preuve de qualité du modèle.

In [6]:
coefficients = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_[0]
}).sort_values("coefficient", key=abs, ascending=False)

print(coefficients)

                  feature  coefficient
3                Monetary    -3.957260
1                 Recency     3.362316
2               Frequency    -0.790601
0                     Age     0.359499
6         Location_Dallas     0.339801
11  Location_Philadelphia     0.281558
5        Location_Chicago     0.238307
12       Location_Phoenix     0.212480
4             Gender_Male    -0.198220
14     Location_San Diego     0.169699
15      Location_San Jose    -0.133113
9    Location_Los Angeles     0.118478
7        Location_Houston     0.103098
10      Location_New York    -0.073832
13   Location_San Antonio     0.071647
8   Location_Jacksonville     0.027163


### Interprétation — coefficients Logistic Regression

Les 3 features RFM dominent très largement, la démographie est négligeable :

- **Monetary (-3.96)** : plus un client dépense, moins il risque de churner
  (coefficient négatif). Le coefficient le plus fort du modèle.
- **Recency (+3.36)** : plus le dernier achat est ancien, plus le risque de
  churn augmente. Cohérent avec la définition métier classique du churn.
- **Frequency (-0.79)** : plus un client achète souvent, moins il risque de
  partir — logique, mais poids nettement plus faible que Monetary/Recency.
- **Age, Gender, Location** : tous sous 0.36 en valeur absolue, quasi
  négligeables comparés aux 3 RFM. Aucune ville ne se détache comme un
  signal fort de churn.

**Confirme que le modèle a appris un signal RFM cohérent et interprétable**,
pas un artefact caché dans l'encodage des villes ou le genre — rassurant
après le score quasi parfait observé précédemment.

### Rappel — Monetary et la distinction avec Total_Spent

Monetary a le coefficient le plus fort du modèle (-3.96). Rappel : bien que
mathématiquement identique à Total_Spent (même formule), Monetary est
retenu car calculé par nous-mêmes avec une fenêtre temporelle connue,
contrairement à Total_Spent qui était une colonne pré-calculée opaque.
Voir décision complète : docs/churn.md, entrée du 10 septembre.